# 03 — Processamento com PySpark
## Pipeline ETL End-to-End — Olist E-commerce Dataset

Etapa de transformação: lê os dados da camada RAW, aplica
limpeza e tipagem, e salva na camada TRUSTED do Data Lake.

**O que acontece nesta etapa:**
- Instalação e configuração do PySpark no Colab
- Leitura dos Parquets da camada RAW
- Correção de tipos (datas, numéricos)
- Tratamento de valores nulos
- Joins entre tabelas
- Criação das tabelas fato e dimensão
- Escrita na camada TRUSTED

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# Iniciar sessão Spark
spark = SparkSession.builder \
    .appName("olist-etl-processing") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"✅ PySpark {spark.version} iniciado")
print(f"   App: {spark.sparkContext.appName}")

✅ PySpark 4.0.2 iniciado
   App: olist-etl-processing


In [2]:
LAKE_PATH   = "/content/data_lake"
RAW_PATH    = f"{LAKE_PATH}/raw"
TRUSTED_PATH = f"{LAKE_PATH}/trusted"

os.makedirs(TRUSTED_PATH, exist_ok=True)

print("✅ Caminhos configurados")
print(f"   RAW:     {RAW_PATH}")
print(f"   TRUSTED: {TRUSTED_PATH}")

✅ Caminhos configurados
   RAW:     /content/data_lake/raw
   TRUSTED: /content/data_lake/trusted


In [3]:
# Verificar se o Data Lake existe, se não, recriar
import os

if not os.path.exists("/content/data_lake/raw/customers"):
    print("⚠️ Data Lake não encontrado — recriando...")

    !pip install kagglehub -q
    import kagglehub
    import pandas as pd
    from datetime import datetime
    import json

    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

    LAKE_PATH  = "/content/data_lake"
    RAW_PATH   = f"{LAKE_PATH}/raw"
    TRUSTED_PATH = f"{LAKE_PATH}/trusted"

    for folder in [RAW_PATH, TRUSTED_PATH, f"{LAKE_PATH}/logs"]:
        os.makedirs(folder, exist_ok=True)

    TABLES = {
        "olist_customers_dataset.csv":           "customers",
        "olist_sellers_dataset.csv":             "sellers",
        "olist_orders_dataset.csv":              "orders",
        "olist_order_items_dataset.csv":         "order_items",
        "olist_order_payments_dataset.csv":      "order_payments",
        "olist_order_reviews_dataset.csv":       "order_reviews",
        "olist_products_dataset.csv":            "products",
        "olist_geolocation_dataset.csv":         "geolocation",
        "product_category_name_translation.csv": "category_translation"
    }

    for csv_file, table_name in TABLES.items():
        df = pd.read_csv(f"{path}/{csv_file}")
        df['_ingested_at'] = datetime.now().isoformat()
        df['_source_file'] = csv_file
        table_path = f"{RAW_PATH}/{table_name}"
        os.makedirs(table_path, exist_ok=True)
        df.to_parquet(f"{table_path}/data.parquet", index=False)
        print(f"   ✅ {table_name:<25} {len(df):>10,} linhas")

    print("\n✅ Data Lake recriado com sucesso!")

else:
    print("✅ Data Lake encontrado — continuando...")

⚠️ Data Lake não encontrado — recriando...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
   ✅ customers                     99,441 linhas
   ✅ sellers                        3,095 linhas
   ✅ orders                        99,441 linhas
   ✅ order_items                  112,650 linhas
   ✅ order_payments               103,886 linhas
   ✅ order_reviews                 99,224 linhas
   ✅ products                      32,951 linhas
   ✅ geolocation                1,000,163 linhas
   ✅ category_translation              71 linhas

✅ Data Lake recriado com sucesso!


In [4]:
print("📖 Lendo camada RAW...")

customers    = spark.read.parquet(f"{RAW_PATH}/customers")
sellers      = spark.read.parquet(f"{RAW_PATH}/sellers")
orders       = spark.read.parquet(f"{RAW_PATH}/orders")
order_items  = spark.read.parquet(f"{RAW_PATH}/order_items")
order_payments = spark.read.parquet(f"{RAW_PATH}/order_payments")
order_reviews  = spark.read.parquet(f"{RAW_PATH}/order_reviews")
products     = spark.read.parquet(f"{RAW_PATH}/products")
geolocation  = spark.read.parquet(f"{RAW_PATH}/geolocation")
category_tr  = spark.read.parquet(f"{RAW_PATH}/category_translation")

tables = {
    "customers": customers, "sellers": sellers,
    "orders": orders, "order_items": order_items,
    "order_payments": order_payments, "order_reviews": order_reviews,
    "products": products, "geolocation": geolocation,
    "category_translation": category_tr
}

print(f"\n{'Tabela':<25} {'Linhas':>12} {'Colunas':>10}")
print("-" * 50)
for name, df in tables.items():
    print(f"   {name:<23} {df.count():>12,} {len(df.columns):>10}")

📖 Lendo camada RAW...

Tabela                          Linhas    Colunas
--------------------------------------------------
   customers                     99,441          7
   sellers                        3,095          6
   orders                        99,441         10
   order_items                  112,650          9
   order_payments               103,886          7
   order_reviews                 99,224          9
   products                      32,951         11
   geolocation                1,000,163          7
   category_translation              71          4


In [5]:
print("🔧 Processando: orders")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean = orders

# Converter colunas de data
for col in date_cols:
    orders_clean = orders_clean.withColumn(
        col, F.to_timestamp(F.col(col))
    )

# Remover pedidos sem status ou sem data de compra
orders_clean = orders_clean.filter(
    F.col("order_status").isNotNull() &
    F.col("order_purchase_timestamp").isNotNull()
)

# Adicionar colunas derivadas úteis
orders_clean = orders_clean \
    .withColumn("purchase_year",  F.year("order_purchase_timestamp")) \
    .withColumn("purchase_month", F.month("order_purchase_timestamp")) \
    .withColumn("purchase_day",   F.dayofmonth("order_purchase_timestamp")) \
    .withColumn("purchase_weekday", F.dayofweek("order_purchase_timestamp")) \
    .withColumn("delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    ) \
    .withColumn("is_late",
        F.when(
            F.col("order_delivered_customer_date") >
            F.col("order_estimated_delivery_date"), 1
        ).otherwise(0)
    )

# Remover colunas de metadados de ingestão
orders_clean = orders_clean.drop("_ingested_at", "_source_file")

print(f"   Antes:  {orders.count():,} linhas")
print(f"   Depois: {orders_clean.count():,} linhas")
print(f"   Colunas adicionadas: purchase_year, purchase_month,")
print(f"                        purchase_day, purchase_weekday,")
print(f"                        delivery_days, is_late")

🔧 Processando: orders
   Antes:  99,441 linhas
   Depois: 99,441 linhas
   Colunas adicionadas: purchase_year, purchase_month,
                        purchase_day, purchase_weekday,
                        delivery_days, is_late


In [6]:
print("🔧 Processando: order_items")

order_items_clean = order_items \
    .withColumn("price",         F.col("price").cast(DoubleType())) \
    .withColumn("freight_value", F.col("freight_value").cast(DoubleType())) \
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date")) \
    .withColumn("total_value",   F.col("price") + F.col("freight_value")) \
    .filter(F.col("price") > 0) \
    .drop("_ingested_at", "_source_file")

print(f"   Linhas:  {order_items_clean.count():,}")
print(f"   Coluna adicionada: total_value (price + freight_value)")

🔧 Processando: order_items
   Linhas:  112,650
   Coluna adicionada: total_value (price + freight_value)


In [7]:
print("🔧 Processando: order_payments")

order_payments_clean = order_payments \
    .withColumn("payment_value", F.col("payment_value").cast(DoubleType())) \
    .withColumn("payment_installments",
                F.col("payment_installments").cast(IntegerType())) \
    .filter(
        F.col("payment_type").isNotNull() &
        (F.col("payment_value") > 0)
    ) \
    .drop("_ingested_at", "_source_file")

print(f"   Linhas: {order_payments_clean.count():,}")

print("\n   Formas de pagamento:")
order_payments_clean.groupBy("payment_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

🔧 Processando: order_payments
   Linhas: 103,877

   Formas de pagamento:
+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5769|
|  debit_card| 1529|
+------------+-----+



In [8]:
print("🔧 Processando: order_reviews")

order_reviews_clean = order_reviews \
    .withColumn("review_score", F.col("review_score").cast(IntegerType())) \
    .withColumn("review_creation_date",
                F.to_timestamp("review_creation_date")) \
    .withColumn("review_answer_timestamp",
                F.to_timestamp("review_answer_timestamp")) \
    .filter(F.col("review_score").isNotNull()) \
    .select(
        "review_id", "order_id", "review_score",
        "review_creation_date", "review_answer_timestamp"
    ) \
    .drop("_ingested_at", "_source_file")

print(f"   Linhas: {order_reviews_clean.count():,}")

avg_score = order_reviews_clean \
    .agg(F.avg("review_score")).collect()[0][0]
print(f"   Nota média: {avg_score:.2f}")

🔧 Processando: order_reviews
   Linhas: 99,224
   Nota média: 4.09


In [9]:
print("🔧 Processando: products")

products_clean = products \
    .join(
        category_tr.select("product_category_name",
                            "product_category_name_english"),
        on="product_category_name",
        how="left"
    ) \
    .withColumn("product_weight_g",
                F.col("product_weight_g").cast(DoubleType())) \
    .withColumn("product_length_cm",
                F.col("product_length_cm").cast(DoubleType())) \
    .withColumn("product_height_cm",
                F.col("product_height_cm").cast(DoubleType())) \
    .withColumn("product_width_cm",
                F.col("product_width_cm").cast(DoubleType())) \
    .fillna("unknown", subset=["product_category_name_english"]) \
    .drop("_ingested_at", "_source_file")

print(f"   Linhas: {products_clean.count():,}")
print(f"   Tradução de categorias aplicada ✅")

🔧 Processando: products
   Linhas: 32,951
   Tradução de categorias aplicada ✅


In [10]:
print("🔧 Processando: geolocation")

# Deduplica por CEP — mantém apenas um registro por prefixo
geolocation_clean = geolocation \
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ) \
    .withColumn("geolocation_lat",
                F.col("geolocation_lat").cast(DoubleType())) \
    .withColumn("geolocation_lng",
                F.col("geolocation_lng").cast(DoubleType())) \
    .dropDuplicates(["geolocation_zip_code_prefix"]) \
    .drop("_ingested_at", "_source_file")

print(f"   Antes (com duplicatas):  1,000,163")
print(f"   Depois (deduplicado):    {geolocation_clean.count():,}")

🔧 Processando: geolocation
   Antes (com duplicatas):  1,000,163
   Depois (deduplicado):    19,015


In [11]:
print("🏗️ Construindo: fato_pedidos")

fato_pedidos = orders_clean \
    .join(order_items_clean.select(
        "order_id",
        F.col("price").alias("item_price"),
        F.col("freight_value").alias("item_freight"),
        F.col("total_value").alias("item_total"),
        "product_id",
        "seller_id"
    ), on="order_id", how="left") \
    .join(order_payments_clean.select(
        "order_id",
        F.col("payment_type").alias("payment_type"),
        F.col("payment_value").alias("payment_value"),
        F.col("payment_installments").alias("installments")
    ), on="order_id", how="left") \
    .join(order_reviews_clean.select(
        "order_id",
        F.col("review_score").alias("review_score")
    ), on="order_id", how="left") \
    .join(customers.select(
        "customer_id",
        F.col("customer_state").alias("customer_state"),
        F.col("customer_city").alias("customer_city")
    ).drop("_ingested_at","_source_file"),
    on="customer_id", how="left")

print(f"   Linhas:  {fato_pedidos.count():,}")
print(f"   Colunas: {len(fato_pedidos.columns)}")
print(f"\n   Schema:")
fato_pedidos.printSchema()

🏗️ Construindo: fato_pedidos
   Linhas:  119,137
   Colunas: 25

   Schema:
root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_weekday: integer (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- is_late: integer (nullable = false)
 |-- item_price: double (nullable = true)
 |-- item_freight: double (nullable = true)
 |-- item_total: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- 

In [12]:
from datetime import datetime

print("💾 Salvando camada TRUSTED...")

trusted_tables = {
    "fato_pedidos":        fato_pedidos,
    "dim_customers":       customers.drop("_ingested_at","_source_file"),
    "dim_sellers":         sellers.drop("_ingested_at","_source_file"),
    "dim_products":        products_clean,
    "dim_geolocation":     geolocation_clean,
    "orders_clean":        orders_clean,
    "order_items_clean":   order_items_clean,
    "order_payments_clean": order_payments_clean,
    "order_reviews_clean": order_reviews_clean
}

results = []
for name, df in trusted_tables.items():
    output = f"{TRUSTED_PATH}/{name}"
    df.write.mode("overwrite").parquet(output)
    count = df.count()
    results.append({"table": name, "rows": count})
    print(f"   ✅ {name:<30} {count:>10,} linhas")

print(f"\n{'='*55}")
print(f"✅ PROCESSAMENTO CONCLUÍDO")
print(f"   Tabelas na camada TRUSTED: {len(results)}")
print(f"{'='*55}")
print(f"\nPRÓXIMA ETAPA: 04_load.ipynb")
print(f"Carga no Data Warehouse (SQL Server)")

💾 Salvando camada TRUSTED...
   ✅ fato_pedidos                      119,137 linhas
   ✅ dim_customers                      99,441 linhas
   ✅ dim_sellers                         3,095 linhas
   ✅ dim_products                       32,951 linhas
   ✅ dim_geolocation                    19,015 linhas
   ✅ orders_clean                       99,441 linhas
   ✅ order_items_clean                 112,650 linhas
   ✅ order_payments_clean              103,877 linhas
   ✅ order_reviews_clean                99,224 linhas

✅ PROCESSAMENTO CONCLUÍDO
   Tabelas na camada TRUSTED: 9

PRÓXIMA ETAPA: 04_load.ipynb
Carga no Data Warehouse (SQL Server)
